In [ ]:
import numpy as np
import torch
from collections import defaultdict
from scipy.sparse import csr_matrix
from recbole.data.interaction import Interaction

def create_item_groups_from_train_recbole(train_data, dataset, short_head_ratio=0.2):
    """
    using RecBole's internal item id（0..item_num-1），
    generating item groups based on train_data interactions ：
      group 1 = short-head, group 2 = long-tail.
    """
    print("\n" + "-"*80)
    print("Creating Item Groups from TRAIN data (Popularity-based, RecBole IDs)")
    print("-"*80)

    item_num = dataset.item_num
    iid_field = dataset.iid_field

    inter_tr = train_data.dataset.inter_feat
    iids = inter_tr[iid_field].cpu().numpy()

    item_popularity = np.zeros(item_num, dtype=np.int64)
    for iid in iids:
        if 0 <= iid < item_num:
            item_popularity[iid] += 1

    sorted_indices = np.argsort(item_popularity)[::-1]

    item_groups = np.ones(item_num, dtype=int) * 2
    n_short_head = int(item_num * short_head_ratio)
    item_groups[sorted_indices[:n_short_head]] = 1
    short_head_pop = item_popularity[item_groups == 1]
    long_tail_pop = item_popularity[item_groups == 2]

    print(f"  Total items (including pad=0): {item_num}")
    print(f"  Short-head items (group 1): {(item_groups == 1).sum()} ({short_head_ratio*100:.1f}%)")
    print(f"  Long-tail items (group 2): {(item_groups == 2).sum()} ({(1-short_head_ratio)*100:.1f}%)")
    print(f"\n  Popularity Statistics (TRAIN only):")
    print(f"    Short-head avg interactions: {short_head_pop.mean():.2f}")
    print(f"    Long-tail avg interactions: {long_tail_pop.mean():.2f}")
    print(f"    Short-head min interactions: {short_head_pop.min():.0f}")
    print(f"    Long-tail max interactions: {long_tail_pop.max():.0f}")

    return item_groups, item_popularity


def full_sort_scores_no_hist(uid_series, model, dataset, device):
    model.eval()
    device = device or torch.device("cpu")

    uid_series = torch.tensor(uid_series, device=device)
    uid_field_local = dataset.uid_field

    input_interaction = dataset.join(Interaction({uid_field_local: uid_series}))
    input_interaction = input_interaction.to(device)

    try:
        scores = model.full_sort_predict(input_interaction)
    except NotImplementedError:
        input_interaction = input_interaction.repeat_interleave(dataset.item_num)
        input_interaction.update(
            dataset.get_item_feature().to(device).repeat(len(uid_series))
        )
        scores = model.predict(input_interaction)

    scores = scores.view(-1, dataset.item_num)
    scores[:, 0] = -np.inf
    return scores


def build_sparse_from_interactions(inter_feat, uid_field, iid_field,
                                   user_offset, item_offset,
                                   user_size, item_size):
    uids = inter_feat[uid_field].cpu().numpy()
    iids = inter_feat[iid_field].cpu().numpy()

    rows = uids - user_offset
    cols = iids - item_offset
    data = np.ones_like(rows, dtype=np.float32)

    mat = csr_matrix(
        (data, (rows, cols)),
        shape=(user_size, item_size),
        dtype=np.float32
    )
    return mat




In [ ]:
import pandas as pd
import os
import numpy as np
# full_sort_topk
from recbole.utils.case_study import full_sort_topk
from recbole.trainer import Trainer
from recbole.quick_start import run_recbole, load_data_and_model
import sys
sys.path.append("..")
from calculate_fairness import (
    compute_all_consumer_metrics,
    compute_all_provider_metrics,
    print_fairness_results
)
from evaluate_utils import computeTopNAccuracy,print_results
os.chdir("./A2GDiffRec/") 
# path to model path
model_path= "./baselines/search_results/ml-1m/MultiVAE/checkpoints/MultiVAE.pth"
config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
        model_file=model_path
    )
model.eval()
trainer = Trainer(config, model)
device = config["device"]
uid_field = config["USER_ID_FIELD"]
iid_field = config["ITEM_ID_FIELD"]

DATASET = config["dataset"]
TOPN = config["topk"]   # e.g. [10, 20, 50, 100]


def evaluate_recbole_with_lists(model, dataset, train_data, valid_data, test_data, topN, eval_config=None):
    _cfg = eval_config if eval_config is not None else config
    _device = _cfg.device if hasattr(_cfg, 'device') else config["device"]

    item_num = dataset.item_num  

    all_inter = dataset.inter_feat
    all_uids = all_inter[uid_field].cpu().numpy()
    u_min, u_max = all_uids.min(), all_uids.max()
    user_offset = u_min
    user_size = u_max - user_offset + 1

    print(f"user_id range: [{u_min}, {u_max}], user_size={user_size}, item_num={item_num}")

    def build_sparse(inter_feat):
        uids = inter_feat[uid_field].cpu().numpy() - user_offset
        iids = inter_feat[iid_field].cpu().numpy() 
        data = np.ones_like(uids, dtype=np.float32)
        return csr_matrix((data, (uids, iids)), shape=(user_size, item_num), dtype=np.float32)

    mat_tr = build_sparse(train_data.dataset.inter_feat)
    mat_va = build_sparse(valid_data.dataset.inter_feat)
    data_te = build_sparse(test_data.dataset.inter_feat)


    mask_tv = mat_tr + mat_va
    mask_tv.data[:] = 1.0

    e_N = user_size
    topk = topN[-1]

    target_items = []
    for i in range(e_N):
        target_items.append(data_te[i, :].nonzero()[1].tolist())

    uid_series = np.arange(user_size, dtype=np.int64) + user_offset

    with torch.no_grad():
        scores = full_sort_scores_no_hist(uid_series, model, dataset, device=_device)
        # scores: [user_size, item_num]

        predict_items = []
        eval_batch_size = _cfg.eval_batch_size if hasattr(_cfg, 'eval_batch_size') else 1024

        for batch_start in range(0, e_N, eval_batch_size):
            batch_end = min(batch_start + eval_batch_size, e_N)
            batch_idx = list(range(batch_start, batch_end))

            prediction = scores[batch_idx, :].clone()   # [B, item_num]

       
            his_data = mask_tv[batch_idx]
            prediction[his_data.nonzero()] = -np.inf

            _, indices = torch.topk(prediction, topk, dim=1)
            predict_items.extend(indices.cpu().numpy().tolist())

    precision, recall, ndcg, mrr = computeTopNAccuracy(target_items, predict_items, topN)

    metrics = {
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "mrr": mrr,
    }

    return metrics, predict_items, target_items, uid_series



metrics, predict_items, target_items, uid_series = evaluate_recbole_with_lists(
    model=model,
    dataset=dataset,
    train_data=train_data,
    valid_data=valid_data,
    test_data=test_data,
    topN=TOPN,
)

print("RecBole baseline metrics (same pipeline as guided_main.py):")
print("precision:", metrics["precision"])
print("recall   :", metrics["recall"])
print("ndcg     :", metrics["ndcg"])
print("mrr      :", metrics["mrr"])
row_data = {}

# Unpack metrics dict into lists, ensure capitalization for keys
precision = metrics["precision"]
recall = metrics["recall"]
NDCG = metrics["ndcg"]
MRR = metrics["mrr"]

for i, k in enumerate(TOPN):
    row_data[f"precision@{k}"] = precision[i]
    row_data[f"recall@{k}"] = recall[i]
    row_data[f"NDCG@{k}"] = NDCG[i]
    row_data[f"MRR@{k}"] = MRR[i]
df = pd.DataFrame([row_data])

short_head_ratio = 0.2 
item_groups, item_popularity = create_item_groups_from_train_recbole(
    train_data, dataset, short_head_ratio=short_head_ratio
)
predicted_indices = np.array(predict_items, dtype=np.int64)  # [num_users, topk_max]
uid_series = np.array(uid_series, dtype=np.int64)       

print("predicted_indices min/max:", predicted_indices.min(), predicted_indices.max())
print("item_groups len:", len(item_groups))

assert predicted_indices.min() >= 0
assert predicted_indices.max() < len(item_groups)

provider_results = compute_all_provider_metrics(
    predicted_indices=predicted_indices,
    item_groups=item_groups,
    short_head_ratio=short_head_ratio,
    topN=TOPN,
    n_items=len(item_groups),
)

print("Provider-side fairness (RecBole baseline):")
print(provider_results)

for key, value in provider_results.items():
    df[key] = value


In [ ]:
print("save to ", save_path+DATASET+config["model"]+str(config["seed"])+".csv")



In [ ]:
# onliy show k=50
df[[col for col in df.columns if '@50' in col]].T

In [ ]:
import pandas as pd

def get_mean_std_at_K(dataset, model, seeds, save_path, K_list=["10", "20", "50", "100"]):
    """
    Read csvs for a given model+dataset+seeds, and return mean & std of metrics for K in K_list.
    Returns:
        summary: pd.DataFrame with rows as metrics (e.g. 'precision@10') 
                 and columns ['mean', 'std']
    """
    df_list = []
    for seed in seeds:
        filename = f"{save_path}{dataset}{model}{seed}.csv"
        df = pd.read_csv(filename)
        df_list.append(df)
    df = pd.concat(df_list, ignore_index=True)
# seed,model,dataset,precision@10,recall@10,NDCG@10,MRR@10,precision@20,recall@20,NDCG@20,MRR@20,precision@50,recall@50,NDCG@50,MRR@50,precision@100,recall@100,NDCG@100,MRR@100,DeltaExposure@10,DeltaVisibility@10,APLT@10,ItemCoverage@10,LongtailCoverage@10,Tail@10,Gini@10,Entropy@10,DeltaExposure@20,DeltaVisibility@20,APLT@20,ItemCoverage@20,LongtailCoverage@20,Tail@20,Gini@20,Entropy@20,DeltaExposure@50,DeltaVisibility@50,APLT@50,ItemCoverage@50,LongtailCoverage@50,Tail@50,Gini@50,Entropy@50,DeltaExposure@100,DeltaVisibility@100,APLT@100,ItemCoverage@100,LongtailCoverage@100,Tail@100,Gini@100,Entropy@100

    metrics = [ "recall", "NDCG", "APLT", "ItemCoverage", "LongtailCoverage", "Entropy","DeltaExposure","Gini"]
    all_results = []
    index_names = []
    for metric in metrics:
        for k in K_list:
            col = f"{metric}@{k}"
            if col in df.columns:
                vals = df[col].values
                all_results.append([vals.mean(), vals.std()])
                index_names.append(col)
    summary = pd.DataFrame(all_results, columns=["mean", "std"], index=index_names)
    return summary

# Example usage:
dataset = config["dataset"]
model = config["model"]
seeds = [config["seed"]]

summary_df = get_mean_std_at_K(dataset, model, seeds, save_path,)
summary_df
#write to csv
#summary_df.to_csv(save_path+dataset+model+"_mean_std.csv", index=True)
